In [2]:
# =========================
# IMPORTS
# =========================
import os

import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [3]:
# =========================
# CONFIG (EASY TO CHANGE)
# =========================
EPOCHS = 5
BATCH_SIZE = 64

# Try multiple configs automatically
WIDTHS = [128, 256, 512]
LAYERS = [3, 5, 10]

DROPOUT = 0.3
LEARNING_RATE = 1e-3
L2 = 1e-4  # weight decay

In [4]:
# =========================
# LOAD DATA
# =========================
df = pd.read_csv("/content/drive/MyDrive/scikit_cleaned.csv")

texts = (df['subject'].fillna('') + " " + df['body'].fillna(''))
labels = df['label']

In [5]:
# =========================
# SPLIT: 60 / 20 / 20
# =========================
X_temp, X_test, y_temp, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42  # 0.25 of 80% = 20%
)

In [6]:
# =========================
# VECTORIZER (BAG OF WORDS)
# =========================
vectorizer = CountVectorizer(max_features=10000)

X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_val_vec = vectorizer.transform(X_val).toarray()
X_test_vec = vectorizer.transform(X_test).toarray()

# Convert to tensors
X_train_tensor = torch.tensor(X_train_vec, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val_vec, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_vec, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

input_size = X_train_tensor.shape[1]

In [8]:
# =========================
# MODEL DEFINITION
# =========================
class PhishingNN(nn.Module):
    def __init__(self, input_size, width, depth):
        super().__init__()

        layers = []

        # input layer
        layers.append(nn.Linear(input_size, width))
        layers.append(nn.GELU())
        layers.append(nn.Dropout(DROPOUT))

        # hidden layers
        for _ in range(depth - 1):
            layers.append(nn.Linear(width, width))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(DROPOUT))

        # output layer
        layers.append(nn.Linear(width, 1))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [9]:
# =========================
# TRAIN FUNCTION
# =========================
def train_model(model, X_train, y_train, X_val, y_val):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=L2)

    for epoch in range(EPOCHS):
        model.train()

        outputs = model(X_train).squeeze()
        loss = criterion(outputs, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Validation
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val).squeeze()
            val_preds = (torch.sigmoid(val_outputs) > 0.5).int()
            val_acc = accuracy_score(y_val, val_preds)

        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss.item():.4f} | Val Acc: {val_acc:.4f}")

    return val_acc


In [10]:
# =========================
# EXPERIMENT LOOP
# =========================
results = []

for width in WIDTHS:
    for depth in LAYERS:
        print(f"\nTraining model: width={width}, depth={depth}")

        model = PhishingNN(input_size, width, depth)

        val_acc = train_model(
            model,
            X_train_tensor,
            y_train_tensor,
            X_val_tensor,
            y_val_tensor
        )

        results.append({
            "width": width,
            "depth": depth,
            "val_acc": val_acc,
            "model": model
        })


Training model: width=128, depth=3
Epoch 1/5 | Loss: 0.6943 | Val Acc: 0.5757
Epoch 2/5 | Loss: 0.6796 | Val Acc: 0.7406
Epoch 3/5 | Loss: 0.6616 | Val Acc: 0.8202
Epoch 4/5 | Loss: 0.6397 | Val Acc: 0.8443
Epoch 5/5 | Loss: 0.6153 | Val Acc: 0.8451

Training model: width=128, depth=5
Epoch 1/5 | Loss: 0.6942 | Val Acc: 0.4914
Epoch 2/5 | Loss: 0.6924 | Val Acc: 0.5252
Epoch 3/5 | Loss: 0.6889 | Val Acc: 0.6473
Epoch 4/5 | Loss: 0.6826 | Val Acc: 0.7960
Epoch 5/5 | Loss: 0.6729 | Val Acc: 0.7795

Training model: width=128, depth=10
Epoch 1/5 | Loss: 0.6947 | Val Acc: 0.4860
Epoch 2/5 | Loss: 0.6943 | Val Acc: 0.4860
Epoch 3/5 | Loss: 0.6939 | Val Acc: 0.4860
Epoch 4/5 | Loss: 0.6935 | Val Acc: 0.4860
Epoch 5/5 | Loss: 0.6933 | Val Acc: 0.4894

Training model: width=256, depth=3
Epoch 1/5 | Loss: 0.6928 | Val Acc: 0.7712
Epoch 2/5 | Loss: 0.6731 | Val Acc: 0.7194
Epoch 3/5 | Loss: 0.6444 | Val Acc: 0.7234
Epoch 4/5 | Loss: 0.6087 | Val Acc: 0.7734
Epoch 5/5 | Loss: 0.5669 | Val Acc: 0.

: 

: 

: 

In [ ]:
# =========================
# BEST MODEL SELECTION
# =========================
best = max(results, key=lambda x: x["val_acc"])

print("\nBest Model:")
print(best["width"], best["depth"], best["val_acc"])

best_model = best["model"]

In [ ]:
def evaluate_on_test():
    best_model.eval()
    with torch.no_grad():
        outputs = best_model(X_test_tensor).squeeze()
        preds = (torch.sigmoid(outputs) > 0.5).int()
        acc = accuracy_score(y_test, preds)

    print(f"TEST ACCURACY: {acc:.4f}")